### 3.2.6. Sensitivity of Local Minima

$$
\nabla_x f(x^*(u),u)=0,
\qquad
\frac{d x^*}{d u}
= -\left[\nabla_{xx}^2 f(x^*(u),u)\right]^{-1}\nabla_{xu}^2 f(x^*(u),u).
$$


**Explanation:**

Sensitivity asks how an optimal solution changes when the data of the problem change. Near a nonsingular local minimum, the gradient equation can be treated as an implicit equation in both the decision variable and the parameter. The inverse Hessian then converts a small parameter perturbation into a first-order prediction of the change in the minimizer. This is why the [condition number](../../01_Foundations/01_Linear_Algebra/07_Theoretical_Linear_Algebra/31_condition_number.ipynb) matters: even when the formula is valid, a poorly conditioned Hessian can amplify small modeling or numerical changes.

The notation $x^*(u)$ means the minimizer depends on a parameter $u$. The first equation keeps the gradient with respect to $x$ equal to zero, and the derivative formula uses a Hessian inverse to measure how the optimizer changes when the parameter changes.

For new optimization students, the key point is that optimality is not only about finding $x^*$ once. Engineering models contain measured constants, approximations, and design parameters, so one must also ask whether the optimizer is stable under small changes. Although this topic is slightly more advanced, it gives important intuition for numerical reliability.

**Intuition:**

A strongly curved quadratic usually has a minimizer that moves smoothly as a parameter varies. By contrast, the singular example in the figure shows behavior at finer and finer scales near a strict minimum, where nearby local minima and maxima accumulate. The code cell demonstrates the regular case by differentiating a simple parameterized stationary equation.

<img src="../../Figures/030107_berts_fig_1_1_8_singular_strict_minimum.jpeg" alt="Bertsekas Figure 1.1.8: singular strict minimum under finer scales" style="width: 60%; height: auto;">


**Numerical Example:**

Unlike the singular behavior in the figure, a nonsingular quadratic minimum moves smoothly. Let
$$
f(x,u)=\frac{1}{2}(x-u)^2+\frac{1}{10}ux.
$$

The stationary equation is
$$
\frac{\partial f}{\partial x}=x-u+\frac{1}{10}u=x-\frac{9}{10}u=0,
$$
so
$$
x^*(u)=\frac{9}{10}u,
\qquad
\frac{dx^*}{du}=\frac{9}{10}.
$$

At $u=2$, the minimizer is
$$
x^*(2)=\frac{9}{10}\cdot 2=1.8.
$$

If $u$ increases by $0.1$, the sensitivity formula predicts
$$
\Delta x^*\approx \frac{9}{10}(0.1)=0.09,
$$
and the exact new minimizer is
$$
x^*(2.1)=\frac{9}{10}\cdot 2.1=1.89=1.8+0.09.
$$


In [1]:
import sympy as sp

variable, parameter = sp.symbols("x u")
objective = sp.Rational(1, 2) * (variable - parameter)**2 + sp.Rational(1, 10) * parameter * variable
stationary_equation = sp.diff(objective, variable)
minimizer = sp.solve(stationary_equation, variable)[0]
sensitivity = sp.diff(minimizer, parameter)
parameter_value = sp.Integer(2)
parameter_perturbation = sp.Rational(1, 10)
nominal_minimizer = minimizer.subs(parameter, parameter_value)
predicted_change = sensitivity * parameter_perturbation
perturbed_minimizer = minimizer.subs(parameter, parameter_value + parameter_perturbation)

print("minimizer =", minimizer)
print("sensitivity =", sensitivity)
print("nominal_minimizer =", nominal_minimizer)
print("predicted_change =", predicted_change)
print("perturbed_minimizer =", perturbed_minimizer)


minimizer = 9*u/10
sensitivity = 9/10
nominal_minimizer = 9/5
predicted_change = 9/100
perturbed_minimizer = 189/100


**Equivalent `casadi` implementation:**

In [2]:
import casadi as ca

variable = ca.SX.sym("x")
parameter = ca.SX.sym("u")
objective = 0.5 * (variable - parameter) ** 2 + 0.1 * parameter * variable
gradient = ca.gradient(objective, variable)
hessian = ca.jacobian(gradient, variable)
minimizer = -ca.substitute(gradient, variable, 0) / hessian
sensitivity = ca.evalf(ca.jacobian(minimizer, parameter))
nominal_minimizer = ca.evalf(ca.substitute(minimizer, parameter, 2))
predicted_change = sensitivity * 0.1
perturbed_minimizer = ca.evalf(ca.substitute(minimizer, parameter, 2 + 0.1))

print("sensitivity =", sensitivity)
print("nominal_minimizer =", nominal_minimizer)
print("predicted_change =", predicted_change)
print("perturbed_minimizer =", perturbed_minimizer)

sensitivity = 0.9
nominal_minimizer = 1.8
predicted_change = 0.09
perturbed_minimizer = 1.89


**References:**

[📘 Bertsekas, D. P. (1999). *Nonlinear Programming* (2nd ed.), Chapter 1 "Unconstrained Optimization". Athena Scientific.](https://www.athenasc.com/nonlinbook.html)

---

[⬅️ Previous: Nonlinear Programming](./05_nonlinear_programming_overview.ipynb) | [Next: Affine Sets and Convex Sets ➡️](../03_Convex_Optimization/01_affine_sets_and_convex_sets.ipynb)

---
